# DMRG 01: MPS and MPO Basics

This notebook introduces the tensor-network objects used by the DMRG driver in this project. The state is stored as a Matrix Product State (MPS), while the Hamiltonian is stored as a Matrix Product Operator (MPO). DMRG combines both objects and optimizes the MPS one two-site block at a time.


## What you will learn

1. How to initialize an `MPS` and inspect its tensor shapes.
2. How to move the MPS orthogonality center with `position`.
3. How to build 1D Ising and Heisenberg MPOs.
4. How to check that each MPO tensor has the expected rank-4 layout.

The examples use `torch.complex128` because some spin operators, such as `Sy`, are naturally complex.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_quantum_simulation():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation")


try:
    qs = _import_quantum_simulation()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    qs = _import_quantum_simulation()

print("Loaded quantum_simulation from", Path(qs.__file__).parent)


Loaded quantum_simulation from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation


In [2]:
import torch
from quantum_simulation import MPS, Ising1DMPOBuilder, Heisenberg1DMPOBuilder

device = "cpu"  # Change to "cuda" after confirming your GPU setup.
dtype = torch.complex128
print(f"Using device={device}, dtype={dtype}")


Using device=cpu, dtype=torch.complex128


## Create and inspect an MPS

An MPS approximates a many-body wavefunction by factorizing it into local site tensors. In this implementation, `MPS` starts from a random right-canonical state with the orthogonality center at site 0.


In [3]:
num_sites = 6
physical_dim = 2
initial_bond_dim = 4

mps = MPS(
    Nsites=num_sites,
    chid=physical_dim,
    initial_bond_dim=initial_bond_dim,
    device=device,
    dtype=dtype,
    seed=123,
)

print(mps)
for site, tensor in enumerate(mps.A):
    print(f"A[{site}] shape = {tuple(tensor.shape)}")


Initializing MPS as a right-canonical random state...
Initialization complete. Ortho center at site 0.
MPS(Nsites=6, chid=2, ortho_center=0, bond_dims=[2, 4, 4, 4, 2, 1])
A[0] shape = (1, 2, 2)
A[1] shape = (2, 2, 4)
A[2] shape = (4, 2, 4)
A[3] shape = (4, 2, 4)
A[4] shape = (4, 2, 2)
A[5] shape = (2, 2, 1)


## Move the orthogonality center

The `position(site_idx)` method changes the canonical form without running an energy optimization step. This is useful when you want to prepare the MPS for local measurements or for custom tensor-network contractions.


In [4]:
mps.position(3)
print(mps)

mps.position(0)
print(mps)



Moving orthogonality center from 0 to 3...
Orthogonality center is now at site 3.
MPS(Nsites=6, chid=2, ortho_center=3, bond_dims=[2, 4, 4, 4, 2, 1])

Moving orthogonality center from 3 to 0...
Orthogonality center is now at site 0.
MPS(Nsites=6, chid=2, ortho_center=0, bond_dims=[2, 4, 4, 4, 2, 1])


## Build a 1D Ising MPO

An MPO stores the Hamiltonian as a list of rank-4 tensors. The convention used by the DMRG driver is `(W_left, W_right, physical_in, physical_out)`. The MPO builders in `quantum_simulation/mpo` return lists that can be passed directly to `DMRG`.


In [5]:
ising_builder = Ising1DMPOBuilder(
    num_sites=num_sites,
    j_coupling=1.0,
    g_field=0.5,
    device=device,
    dtype=dtype,
)
ising_mpo = ising_builder.get_mpo()

ising_builder.display_bond_dimensions()
for site, tensor in enumerate(ising_mpo):
    print(f"Ising MPO[{site}] shape = {tuple(tensor.shape)}")


1D Transverse-Field Ising Model MPO
N=6, J=1.0, g=0.5

Bond Dimensions:  1 3 3 3 3 3 1

Ising MPO[0] shape = (1, 3, 2, 2)
Ising MPO[1] shape = (3, 3, 2, 2)
Ising MPO[2] shape = (3, 3, 2, 2)
Ising MPO[3] shape = (3, 3, 2, 2)
Ising MPO[4] shape = (3, 3, 2, 2)
Ising MPO[5] shape = (3, 1, 2, 2)


## Build a 1D Heisenberg MPO

The Heisenberg builder exposes independent `Jx`, `Jy`, and `Jz` couplings. This makes it easy to represent isotropic or anisotropic spin chains with the same workflow.


In [6]:
heisenberg_builder = Heisenberg1DMPOBuilder(
    num_sites=num_sites,
    j_coupling_x=1.0,
    j_coupling_y=1.0,
    j_coupling_z=1.0,
    device=device,
    dtype=dtype,
)
heisenberg_mpo = heisenberg_builder.get_mpo()

heisenberg_builder.display_bond_dimensions()
print("Virtual bond dimensions:")
print([tuple(tensor.shape[:2]) for tensor in heisenberg_mpo])


1D Heisenberg Model MPO
N=6, Jx=1.0, Jy=1.0, Jz=1.0

Bond Dimensions:  1 4 5 5 5 4 1

Virtual bond dimensions:
[(1, 4), (4, 5), (5, 5), (5, 5), (5, 4), (4, 1)]


## Takeaways

The basic DMRG inputs are now in place: an `MPS` object for the variational state and an MPO tensor list for the Hamiltonian. The next notebook uses these two objects with `Sweeps` and `DMRG` to run a ground-state optimization.
